# Train Merchant Agent on ChargebackOps

End-to-end GRPO training for the merchant-side chargeback agent.

- Environment: `ChargebackOpsEnvironment` (multi-round adversarial Issuer, arbitration ROI).
- Text interface: `training.env_adapter` (prompt build, completion parse).
- Reward: `training.reward_adapter.compute_reward` — returns the normalised episode score in `[0, 1]`.
- Trainer: `trl.GRPOTrainer` on a small base model so this fits a free Colab T4.
- Checkpoint evaluation + curve plot: `training.curve` (overall + per-difficulty family).

Run order: setup → sanity check → load model → build multi-task prompt dataset (cycles all 11 headline tasks) → 200-step GRPO (saves at 0 / 50 / 100 / 150 / 200) → evaluate each checkpoint → overall curve → per-family curve → ablation table. Both PNGs land in `docs/figures/`.

## 1. Colab setup

Installs TRL, transformers, and the ChargebackOps package itself. Skip if the environment already has them.

In [ ]:
%%capture
import sys
if 'google.colab' in sys.modules:
    # Qwen3.5-0.8B requires bleeding-edge transformers (multimodal VLM, released 2026).
    # TRL >= 0.14 needed for transformers main compatibility.
    !pip install --quiet --upgrade "transformers @ git+https://github.com/huggingface/transformers.git@main"
    !pip install --quiet --upgrade "trl>=0.14" accelerate peft bitsandbytes matplotlib datasets pydantic
    !git clone https://github.com/MitudruDutta/chargebackops.git /content/chargebackops
    %cd /content/chargebackops
    !pip install --quiet -e .

## 2. Sanity-check the env adapter

Run one scripted episode via the text adapter to confirm prompts render and rewards land inside `[0, 1]`.

In [ ]:
from training.env_adapter import build_prompt
from training.reward_adapter import run_episode_with_text_policy

def heuristic_text_policy(prompt: str) -> str:
    return ''  # forces the scripted-heuristic fallback

result = run_episode_with_text_policy('goods_not_received_easy', heuristic_text_policy)
print('score', result.score, 'steps', result.steps_used, 'invalid', result.invalid_actions)

## 3. Load a small base model

`Qwen/Qwen3.5-0.8B` (released 2026) — Qwen3.5 small, 0.8B params, 24 layers, 256K native context, hybrid thinking / non-thinking modes. Fits a free Colab T4 (~6 GB in bf16, ~10 GB with GRPO activations and rollouts at K=4). Multimodal VLM, but we only exercise the text head — JSON action output, no images.

Default mode is non-thinking (matches our structured-JSON action contract). If you flip to thinking mode the completion tokens explode, blowing the prompt-budget assumptions in `compute_reward`.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'Qwen/Qwen3.5-0.8B'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map='auto',
    trust_remote_code=True,
)

## 4. Build the training prompt dataset

GRPO generates K completions per prompt internally and scores each with `compute_reward`. Sample prompts from fresh environment resets across the headline catalog.

In [ ]:
from datasets import Dataset
from scenarios.simulation import list_tasks
from server.chargeback_ops_environment import ChargebackOpsEnvironment
from training.env_adapter import build_prompt

def sample_prompts(n: int = 64):
    tasks = list_tasks()
    prompts, task_ids = [], []
    for i in range(n):
        task = tasks[i % len(tasks)]
        env = ChargebackOpsEnvironment()
        obs = env.reset(task_id=task.task_id).model_dump()
        prompts.append(build_prompt(obs))
        task_ids.append(task.task_id)
    return Dataset.from_dict({'prompt': prompts, 'task_id': task_ids})

train_dataset = sample_prompts(64)
len(train_dataset)

## 5. GRPO training — 200 steps

Saves checkpoints every 50 steps into `./grpo-merchant-agent/checkpoint-*`. Start with `max_steps=1` the first time you run this cell to confirm the gradient path closes, then bump to 200 for the real curve.

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from training.reward_adapter import compute_reward

def reward_fn(prompts, completions, **kwargs):
    task_ids = kwargs.get('task_id') or kwargs.get('task_ids')
    return compute_reward(prompts, completions, task_ids=task_ids)

# TRL >= 0.14: beta=0 by default skips KL ref model (saves ~0.8B params of VRAM).
# processing_class kwarg dropped — pass tokenizer via processing_class only if your
# TRL pin still requires it; latest API auto-resolves from the model.
config = GRPOConfig(
    output_dir='./grpo-merchant-agent',
    per_device_train_batch_size=2,
    num_generations=4,
    max_prompt_length=1024,
    max_completion_length=128,
    learning_rate=5e-6,
    max_steps=200,
    logging_steps=10,
    save_steps=50,
    save_total_limit=5,
    gradient_accumulation_steps=1,
    bf16=torch.cuda.is_available(),
    report_to='none',
    beta=0.0,
)
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[reward_fn],
    args=config,
    train_dataset=train_dataset,
)
trainer.train()

## 6. Evaluate every checkpoint

Loads each saved checkpoint as a text policy and scores it across the headline catalog. Step 0 uses the untrained base model.

In [ ]:
import glob
import re

import torch
from transformers import AutoModelForCausalLM

from training.curve import evaluate_checkpoint

def make_text_policy(ckpt_path: str):
    ckpt_model = AutoModelForCausalLM.from_pretrained(
        ckpt_path,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map='auto',
    )

    def _policy(prompt: str) -> str:
        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1024).to(ckpt_model.device)
        with torch.no_grad():
            out = ckpt_model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False,
                temperature=0.0,
                pad_token_id=tokenizer.pad_token_id,
            )
        return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    return _policy

def checkpoint_step(path: str) -> int:
    match = re.search(r'checkpoint-(\d+)$', path)
    return int(match.group(1)) if match else -1

checkpoint_dirs = sorted(
    glob.glob('./grpo-merchant-agent/checkpoint-*'),
    key=checkpoint_step,
)
checkpoints = [evaluate_checkpoint(step=0, policy=make_text_policy(MODEL_ID))]
for ckpt_dir in checkpoint_dirs:
    step = checkpoint_step(ckpt_dir)
    if step <= 0:
        continue
    checkpoints.append(evaluate_checkpoint(step=step, policy=make_text_policy(ckpt_dir)))
for ckpt in checkpoints:
    print(f'step={ckpt.step:4d}  mean={ckpt.mean_score:.4f}')

## 7. Plot the training curve

Saves the figure to `docs/figures/training_curve.png`. Baseline scores (`heuristic`, `concede_all`, `naive`) overlay as dashed lines for grounding.

In [ ]:
import os

from runners.benchmark_runner import run_policy_sweep
from training.curve import plot_training_curve

os.makedirs('docs/figures', exist_ok=True)
baselines = {s.policy: s.mean_score for s in run_policy_sweep().policies}
plot_training_curve(
    checkpoints,
    'docs/figures/training_curve.png',
    baseline_scores={
        'heuristic': baselines['heuristic'],
        'concede_all': baselines['concede_all'],
        'naive': baselines['naive'],
    },
)
print('saved', 'docs/figures/training_curve.png')

## 8. Ablation table

Compare untrained / heuristic / trained on the same catalog and print a markdown-ready table for `docs/RESULTS.md`.

In [ ]:
untrained = checkpoints[0]
trained = checkpoints[-1]
heuristic_mean = baselines['heuristic']
print('| Agent | Mean score |')
print('| --- | --- |')
print(f'| untrained (step 0)   | {untrained.mean_score:.4f} |')
print(f'| heuristic baseline   | {heuristic_mean:.4f} |')
print(f'| trained (step {trained.step}) | {trained.mean_score:.4f} |')

## 9. Per-family training curve (multi-task RL)

The aggregate curve hides whether GRPO improves uniformly or only on cheap easy tasks. Re-evaluate each checkpoint grouped by difficulty (`easy` / `medium` / `hard` / `nightmare`) and overlay the per-cohort heuristic floor from `run_multi_seed`. Each family's solid line should rise; if `easy` jumps but `nightmare` stays flat, the policy is overfitting to the cheap end of the catalog.

In [ ]:
from collections import defaultdict
from runners.benchmark_runner import run_multi_seed
from training.curve import evaluate_checkpoint_by_family, plot_training_curve_by_family

# Re-eval each checkpoint, but bucket per difficulty.
grouped_checkpoints = []
grouped_checkpoints.append(evaluate_checkpoint_by_family(step=0, policy=make_text_policy(MODEL_ID)))
for ckpt_dir in checkpoint_dirs:
    step = checkpoint_step(ckpt_dir)
    if step <= 0:
        continue
    grouped_checkpoints.append(
        evaluate_checkpoint_by_family(step=step, policy=make_text_policy(ckpt_dir))
    )

# Per-difficulty heuristic floors from the 28-task multi-seed grid.
grid = run_multi_seed(seeds=[7, 17, 31, 42, 53, 77, 99], difficulties=['easy', 'medium', 'hard', 'nightmare'])
heur_by_diff = defaultdict(list)
for ps in grid.policies:
    if ps.policy != 'heuristic':
        continue
    for tr in ps.tasks:
        heur_by_diff[tr.task_id.split('_')[1]].append(tr.score)
baseline_per_family = {
    diff: {'heuristic': sum(scores) / len(scores)}
    for diff, scores in heur_by_diff.items()
}

plot_training_curve_by_family(
    grouped_checkpoints,
    'docs/figures/training_curve_by_family.png',
    baseline_scores=baseline_per_family,
    family_order=['easy', 'medium', 'hard', 'nightmare'],
)
print('saved docs/figures/training_curve_by_family.png')
for g in grouped_checkpoints:
    fam_str = '  '.join(f'{f}={g.by_family[f].mean_score:.3f}' for f in ['easy', 'medium', 'hard', 'nightmare'] if f in g.by_family)
    print(f'step={g.step:4d}  overall={g.overall_mean:.4f}  {fam_str}')